In [1]:
import os
import pandas as pd
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
import nltk
import spacy

In [7]:
folder_path = r"C:\Projects\CodeMix\data\Raw"  
all_data = []


for file in os.listdir(folder_path):
    if file.endswith(".json"):
        file_path = os.path.join(folder_path, file)
        
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f) 
            all_data.extend(data)  

# Convert to DataFrame
df = pd.DataFrame(all_data)
print(df.head()) 
   id                                               text
0   0  Game next level కి వెళ్లాలి, strategy change చ...
1   1    ఈ song vibe చాలా different, beats super catchy!
2   2   Internet speed చాలా slow, video buffer అవుతోంది.
3   3          Traffic చాలా ఎక్కువ, route change చేయాలి.
4   4  ఈ update చాలా useful, features impressive ఉన్న...


   id                                               text
0   0  Game next level కి వెళ్లాలి, strategy change చ...
1   1    ఈ song vibe చాలా different, beats super catchy!
2   2   Internet speed చాలా slow, video buffer అవుతోంది.
3   3          Traffic చాలా ఎక్కువ, route change చేయాలి.
4   4  ఈ update చాలా useful, features impressive ఉన్న...


In [9]:
print(len(df))  # Number of rows


7000000


In [11]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from indicnlp.tokenize import sentence_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

In [35]:
def CleanText(files_path, output_file, cleaned_output_file):
    """This Function is part of text preprocessing and cleaning
        includes removing certain regex patterns, dates, days, months, stopwords,stemming, lemminaization, punctuation,removing numbers 
        """
    text = ""
    with open(output_file, 'r', encoding='utf-8') as file:
        text = file.read()
    patterns = [
    r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Email
    r'(\+?\d{1,4}[\s-])?(?:\(\d{1,3}\)[\s-]?)?\d{1,4}[\s-]?\d{1,4}[\s-]?\d{1,9}',  # Phone numbers
    r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b',  # Dates
    r'https?://[^\s/$.?#].[^\s]*',  # URLs
    r'\[(\d+)\]',  # Bracketed numbers
    r'10.\d{4,9}/[-._;()/:A-Z0-9]+',  # DOIs
    r'(?:ISBN(?:-13)?:?\s*)?(?=[-0-9]{13}$|(?=(?:[-0-9]{17}$)|(?:[-0-9X]{10}$))(?:97[89][-0-9]{10}$))\d{1,5}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?(?:\d|X)',  # ISBNs
    r'[-+]?\d*\.?\d+([eE][-+]?\d+)?',  # Numbers and scientific notation
    r'>.*\n([A-Z\n]+)',  # Block caps
    r'\$\$.*?\$\$',  # LaTeX equations
    r'^([a-zA-Z0-9_\-]+)\.([a-zA-Z0-9]+)Text\sis\sbeen\sEntered:\s:\s$',  # Custom pattern
    r'#\w+',  # Hashtags
    r'@\w+',  # Mentions
    r'\b(?:\d{1,2} [A-Za-z]{3,9} \d{4}|\d{4}/\d{2}/\d{2})\b',  # Dates (expanded formats)
    r'\b(?:\d{1,3}\.){3}\d{1,3}\b',  # IP addresses
    r'<.*?>',  # HTML tags
    r'\((.*?)\)',  # Bracketed content
    r'\b(?:\$|€|₹|£)\d+(?:\.\d{1,2})?\b',  # Currency values
    r'[^\x00-\x7F]+',  # Non-ASCII characters
    r'\b(?:\d{4}[- ]?){3}\d{4}\b',  # Credit card numbers
    r'\b[0-9A-Fa-f]{8}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{12}\b',  # UUIDs
    r'(.)\1{2,}',  # Repeated characters
    r'(?:[A-Za-z]:)?(?:\\[A-Za-z0-9_.-]+)+\\?',  # File paths
    r'\b0[xX][0-9a-fA-F]+\b',  # Hexadecimal numbers
    r'\b[A-Z]+\b',  # All caps
    r'["\'](.*?)["\']',  # Quoted strings
    r'\b[A-Z][a-z]*\b'  # Extract capitalized words
]

    for pattern in patterns:
        text = re.sub(pattern, '', text) # remove regex patterns

    words = text.split()
    months = ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december'] 
    stops = set(stopwords.words('english'))
    stops.update(months) # combing stopwords and months from list to remove them collectively

    # Initialize the stemmer and lemmatizer
    stemmer = PorterStemmer()                          # converts a word to its stem form like running to run 
    lemmatizer = WordNetLemmatizer()                   # converts a word to its dictionary form for no duplicates 

    final = [lemmatizer.lemmatize(stemmer.stem(word)) for word in words if word.lower() not in stops]

    final_text = " ".join(final)
    final_text = final_text.translate(str.maketrans("", "", string.punctuation))
    final_text = "".join([i for i in final_text if not i.isdigit()])
    while "  " in final_text:
        final_text = final_text.replace("  ", " ")

    # Write to a  cleaned text to the new output file
    with open(cleaned_output_file, 'w', encoding='utf-8') as file:
        file.write(final_text)

    return final_text

In [37]:
df["text"] = df["text"].apply(CleanText)

print(df.head()) 


TypeError: CleanText() missing 2 required positional arguments: 'output_file' and 'cleaned_output_file'

In [56]:
import nlu
nlu.load("te.stopwords").predict("""మీరు నన్ను కంటే మెరుగైనది కాదు""")

You need Pyspark installed to run NLU. Run <pip install pyspark==3.0.2>


ImportError: You ned to install Pyspark to run nlu. Run pip install pyspark==3.0.1

In [58]:
import nlu

# Load the Telugu stopwords model
nlu_te = nlu.load("te.stopwords")

# Example Telugu sentence
text = "మీరు నన్ను కంటే మెరుగైనది కాదు"

# Predict stopwords
stopwords = nlu_te.predict(text)

print("Telugu Stopwords:", stopwords)


You need Pyspark installed to run NLU. Run <pip install pyspark==3.0.2>


ImportError: You ned to install Pyspark to run nlu. Run pip install pyspark==3.0.1

In [62]:
from sentence_transformers import SentenceTransformer

# Load multilingual embedding model
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Code-mixed example
sentence1 = "help me"
sentence2 = "నన్ను సహాయం చేయండి"

# Get embeddings
emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)

# Compare similarity (Cosine similarity)
from numpy import dot
from numpy.linalg import norm

cos_sim = dot(emb1, emb2) / (norm(emb1) * norm(emb2))
print("Similarity:", cos_sim)  # Should be high if embeddings are in the same space


model.safetensors:  18%|#7        | 199M/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity: 0.97692436
